# Used Car Price Prediction

Predicting the resale price of used cars using the [Cars24](https://www.cars24.com/) combined listings dataset (`cars_24_combined.csv`).

**Pipeline:**
1. Load and clean the raw listings data
2. Explore relationships between price and key features (year, distance driven, fuel type, etc.)
3. Encode categorical features
4. Train and compare three regression models: Linear Regression, Ridge Regression, and XGBoost
5. Tune hyperparameters with [Optuna](https://optuna.org/) and compare final test performance

**Target variable:** `Price`


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## 1. Load the data

Reads the combined Cars24 listings CSV. See the README for where to get this file.

In [ ]:
data = pd.read_csv('cars_24_combined.csv')
data.head(3)

In [ ]:
print('Shape:', data.shape)
data.info()

## 2. Data Cleaning

Check for missing values, drop rows with missing key identifiers, and fill in what can reasonably be imputed.

In [ ]:
data.isnull().sum()

In [ ]:
# Drop rows where we can't identify the car or its age
before = data.shape
data = data.dropna(subset=['Car Name', 'Year'])
print('Before dropna:', before)
print('After dropna:', data.shape)

In [ ]:
# Location has some missing values with no clear default — mark them explicitly
data['Location'] = data['Location'].fillna('unknown')
data.isnull().sum()

In [ ]:
# Convert model Year into car age (more directly useful as a predictor than a raw year)
current_year = 2026
data['Year'] = current_year - data['Year']
data = data.drop(columns='Unnamed: 0')
data.head(3)

## 3. Exploratory Data Analysis

A quick look at the distribution of key variables and their relationship with price.

In [ ]:
sns.countplot(x='Fuel', data=data)
plt.title('Count of listings by fuel type')
plt.show()

In [ ]:
sns.boxplot(x='Type', data=data)
plt.title('Distribution of car body type')
plt.show()

In [ ]:
sns.boxplot(x=data['Price'])
plt.title('Price distribution (check for outliers)')
plt.show()

In [ ]:
sns.boxplot(x=data['Distance'])
plt.title('Distance driven distribution (check for outliers)')
plt.show()

In [ ]:
corr = data[['Year', 'Distance', 'Owner', 'Price']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation between numeric features and Price')
plt.show()

In [ ]:
sns.scatterplot(x='Year', y='Price', data=data)
plt.title('Car age vs. Price')
plt.show()

In [ ]:
sns.scatterplot(x='Distance', y='Price', data=data)
plt.title('Distance driven vs. Price')
plt.show()

## 4. Feature Encoding

- `Location`: frequency encoding (rare locations get a smaller weight than common ones)
- `Brand`: extracted from the car name, then one-hot encoded
- `Fuel`, `Drive`, `Type`: one-hot encoded

In [ ]:
freq_encoding = data['Location'].value_counts().to_dict()
data['Location'] = data['Location'].map(freq_encoding)

In [ ]:
data['Brand'] = data['Car Name'].str.split().str[0]
data = data.drop(columns=['Car Name'])

data = pd.get_dummies(
    data,
    columns=['Brand', 'Fuel', 'Drive', 'Type'],
    drop_first=True,
    dtype=int
)

data.head()

## 5. Model Training

Train/test split, then baseline Linear Regression, Ridge Regression, and XGBoost models before any tuning.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

x = data.drop(columns=['Price'])
y = data['Price']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)


def report(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f'{name}: R2={r2:.4f}  RMSE={rmse:,.2f}  MAE={mae:,.2f}')
    return rmse

### 5.1 Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

linr = LinearRegression(fit_intercept=True, n_jobs=-1)
linr.fit(x_train, y_train)
y_pred = linr.predict(x_test)
report(y_test, y_pred, 'LinearRegression (baseline)')

### 5.2 Ridge Regression

In [ ]:
from sklearn.linear_model import Ridge

ridg = Ridge(alpha=0.5, fit_intercept=True)
ridg.fit(x_train, y_train)
ridg_pred = ridg.predict(x_test)
report(y_test, ridg_pred, 'Ridge (baseline)')

### 5.3 XGBoost

In [ ]:
import xgboost as xgb

xgb_regressor = xgb.XGBRegressor(
    n_estimators=100,              # number of sequential trees to build
    max_depth=6,                   # max depth of each tree
    learning_rate=0.1,             # step size shrinkage (eta)
    objective='reg:squarederror',
    random_state=42
)
xgb_regressor.fit(x_train, y_train)
xgb_pred = xgb_regressor.predict(x_test)
report(y_test, xgb_pred, 'XGBoost (baseline)')

## 6. Hyperparameter Tuning with Optuna

For each model, run a 50-trial (10 for Linear Regression, which has almost no knobs to tune) Optuna study using
5-fold cross-validated RMSE on the training set only, then refit the best configuration and evaluate on the held-out test set.

In [ ]:
# !pip install optuna  # uncomment if optuna isn't already installed

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
def objective_ridge(trial):
    alpha = trial.suggest_float('alpha', 1e-3, 10.0, log=True)
    fit_intercept = trial.suggest_categorical('fit_intercept', [True, False])
    solver = trial.suggest_categorical('solver', ['auto', 'svd', 'cholesky', 'lsqr'])

    ridge = Ridge(alpha=alpha, fit_intercept=fit_intercept, solver=solver, random_state=42)

    scores = cross_val_score(
        ridge, x_train, y_train, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1
    )
    return -scores.mean()

study_ridge = optuna.create_study(
    direction='minimize', study_name='ridge_tuning',
    sampler=optuna.samplers.TPESampler(seed=42)
)
study_ridge.optimize(objective_ridge, n_trials=50, show_progress_bar=True)

print('Best Ridge parameters:', study_ridge.best_params)
print('Best Ridge CV RMSE:', study_ridge.best_value)

best_ridge = Ridge(**study_ridge.best_params, random_state=42)
best_ridge.fit(x_train, y_train)
ridge_test_rmse = report(y_test, best_ridge.predict(x_test), 'Ridge (tuned)')

In [ ]:
def objective_linreg(trial):
    # LinearRegression only really has this one meaningful knob
    fit_intercept = trial.suggest_categorical('fit_intercept', [True, False])
    linr = LinearRegression(fit_intercept=fit_intercept, n_jobs=-1)

    scores = cross_val_score(
        linr, x_train, y_train, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1
    )
    return -scores.mean()

study_linreg = optuna.create_study(
    direction='minimize', study_name='linreg_tuning',
    sampler=optuna.samplers.TPESampler(seed=42)
)
# Only 2 possible values for fit_intercept, so a small trial budget is enough
study_linreg.optimize(objective_linreg, n_trials=10, show_progress_bar=True)

print('Best LinearRegression parameters:', study_linreg.best_params)
print('Best LinearRegression CV RMSE:', study_linreg.best_value)

best_linr = LinearRegression(**study_linreg.best_params, n_jobs=-1)
best_linr.fit(x_train, y_train)
linreg_test_rmse = report(y_test, best_linr.predict(x_test), 'LinearRegression (tuned)')

In [ ]:
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }

    xgb_model = xgb.XGBRegressor(
        **params,
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1
    )

    scores = cross_val_score(
        xgb_model, x_train, y_train, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1
    )
    return -scores.mean()

study_xgb = optuna.create_study(
    direction='minimize', study_name='xgb_tuning',
    sampler=optuna.samplers.TPESampler(seed=42)
)
study_xgb.optimize(objective_xgb, n_trials=50, show_progress_bar=True)

print('Best XGBoost parameters:', study_xgb.best_params)
print('Best XGBoost CV RMSE:', study_xgb.best_value)

best_xgb = xgb.XGBRegressor(
    **study_xgb.best_params,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)
best_xgb.fit(x_train, y_train)
xgb_test_rmse = report(y_test, best_xgb.predict(x_test), 'XGBoost (tuned)')

## 7. Results

In [ ]:
results = pd.DataFrame({
    'Model': ['Ridge', 'LinearRegression', 'XGBoost'],
    'Best CV RMSE': [study_ridge.best_value, study_linreg.best_value, study_xgb.best_value],
    'Test RMSE': [ridge_test_rmse, linreg_test_rmse, xgb_test_rmse]
}).sort_values('Test RMSE').reset_index(drop=True)

results

In [ ]:
best_model_name = results.loc[0, 'Model']
print(f"Best model on held-out test RMSE: {best_model_name}")
print("XGBoost is expected to lead since it captures non-linear interactions between features "
      "(e.g. brand, age, and distance together) that the linear models can't represent.")